**Author**: Felipe Matheus  
**Start Date**: --/05/2026  
**End Date**: --/--/2026  
**Related Links**:

- Results so far: https://docs.google.com/spreadsheets/d/1OnIWY_bKxgdGU3trECDTL971WfXFPWgbW-dQzgP6KhI/edit?gid=0#gid=0
- Dataset used: https://drive.google.com/drive/u/0/folders/1VsU-taxtwfmDLUu5Z4JmdgXSoxUoB8FI

**Objective**

Build an uncertainty-aware probabilistic predictor of ultimate tensile strength (UTS) after a single cold-drawing pass — the **one-step-ahead** evaluation regime. Each row is treated independently; inputs are the true measured pre-pass state. The multi-step rollout evaluation (the model used recursively along a wire's full pass sequence) lives in the companion notebook `cold_drawing_uts_multi_step.ipynb`.

**Why two notebooks**

- *One-step-ahead* (this notebook): metric on individual predictions with real inputs. Diagnoses the quality of the surrogate as an atomic function.
- *Multi-step rollout* (other notebook): metric on full-wire trajectories where each pass feeds the previous prediction back as input. Diagnoses error compounding and matches how the MBC actually uses the surrogate.

Both are needed; they answer different questions.

**Pipeline**

1. Setup and global configuration.
2. Load + preprocess data (cold drawing has correlated rows within a wire).
3. Train **Model A** (mean predictor) with **GroupKFold** bagging by experiment.
4. Extract OOF predictions per base learner and recover ensemble weights via NNLS.
5. Compute weighted $\hat\mu$ and $\hat\sigma^2_{\text{epist}}$.
6. Build aleatoric targets $\tilde r^2 = \max(r^2_{\text{OOF}} - \hat\sigma^2_{\text{epist}}, 0)$.
7. Train **Model B** in log-space.
8. Calibration diagnostics and scalar recalibration.
9. One-step inference example.

# 1. Setup

In [ ]:
import os
import sys
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

module_path = os.path.abspath(os.path.join('../..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.processing.Processing import Processing
from src.feature_engineering.FeatureEngineering import FeatureEngineering
from src.feature_engineering.CDHelper import CDHelper
from src.modeling.Modeling import Modeling
from src.metrics.Evaluation import Evaluation
from config.Variables import Variables

%load_ext autoreload
%autoreload 2

In [ ]:
proc = Processing()
feng = FeatureEngineering()
modl = Modeling()
evla = Evaluation()
varv = Variables()

## 1.1 Global configuration

All knobs in one place. Adapting this notebook to another cold-drawing surrogate (IACS or grain size) is a single-cell edit.

In [ ]:
# ---- Paths ----
PATH_DATA_RAW = '../../data/raw'
PATH_MODELS = '../../models'
RAW_DATASET_FILE_NAME = 'dataset_cold_drawing_uts.csv'
PROCESS = 'cold_drawing_uts'

# ---- Dataset schema ----
TARGET = 'tensile_strength_final'
FEATURES = [
    'purity',
    'original_tensile_strength',
    'pass_number',
    'original_diameter',
    'initial_diameter',
    'final_diameter',
    'total_strain',
    'reduction_ratio',
    'initial_tensile_strength',
]
GROUP_COL = 'group_experiment_id'   # rows of the same wire share this id

# ---- AutoGluon knobs (Model A) ----
PRESETS_A = 'medium_quality'
NUM_BAG_FOLDS_A = 5
NUM_BAG_SETS_A = 1
NUM_STACK_LEVELS_A = 0
TIME_LIMIT_A = 180

# ---- AutoGluon knobs (Model B) ----
PRESETS_B = 'medium_quality'
NUM_BAG_FOLDS_B = 5
NUM_STACK_LEVELS_B = 0
TIME_LIMIT_B = 90

# ---- Uncertainty pipeline knobs ----
USE_WEIGHTED_VARIANCE = True
VARIANCE_FLOOR_FRAC = 0.01
CALIBRATION_ALPHAS = (0.5, 0.8, 0.9, 0.95)
RECALIBRATION_TARGET_ALPHA = 0.9

# 2. Data

In [ ]:
# Raw CSV — preprocessing follows the existing cold-drawing helper functions.
path = os.path.join(PATH_DATA_RAW, RAW_DATASET_FILE_NAME)
df_raw = pd.read_csv(path).dropna()
df_raw.loc[df_raw['purity'] == '-', 'purity'] = 99.9
df_raw.purity = df_raw.purity.astype(float)

df_grouped = CDHelper.set_group_experiment(df=df_raw)
df_grouped_fixed, npass_report = CDHelper.check_n_pass(df_grouped, correct=True)
df_full = CDHelper.add_initial_tensile_strength(df_grouped_fixed)

# Keep only sanity-checked rows.
df = df_full[df_full.sanity_check_ok == True].copy()
df = df[FEATURES + [TARGET, GROUP_COL]].reset_index(drop=True)

assert df[FEATURES + [TARGET]].isna().sum().sum() == 0, 'NaNs in inputs'
print(f'Dataset shape: {df.shape}')
print(f'Number of unique wires (groups): {df[GROUP_COL].nunique()}')
print(f'Passes per wire (describe):')
print(df.groupby(GROUP_COL).size().describe())

**Critical note on GroupKFold.** Each wire (`group_experiment_id`) contributes multiple correlated rows — one per pass. Plain K-fold cross-validation leaks information between train and test folds, because rows of the same wire end up in both. We use `GroupKFold` semantics via AutoGluon's `groups` argument.

# 3. Train Model A — Mean Predictor with Group-Aware Bagging

Bagging is non-optional: without it, no honest OOF predictions can be extracted. Passing the `group_col` keeps all rows of a wire in the same fold.

In [ ]:
predictor_a = modl.fit_model_a_grouped(
    df=df,
    target=TARGET,
    features=FEATURES,
    group_col=GROUP_COL,
    path=os.path.join(PATH_MODELS, PROCESS, 'model_a'),
    presets=PRESETS_A,
    num_bag_folds=NUM_BAG_FOLDS_A,
    num_bag_sets=NUM_BAG_SETS_A,
    num_stack_levels=NUM_STACK_LEVELS_A,
    time_limit=TIME_LIMIT_A,
)

In [ ]:
predictor_a.leaderboard(silent=True)

# 4. OOF Predictions and Ensemble Weights

We need three objects to compute the epistemic variance:
1. OOF predictions of the WeightedEnsemble (the final model).
2. OOF predictions of each base learner individually (matrix M × N).
3. The ensemble weights, recovered via NNLS from quantities AutoGluon exposes.

In [ ]:
mu_oof = predictor_a.predict_oof()        # OOF of the WeightedEnsemble_L2
residuals_oof = df[TARGET] - mu_oof.values
print(f'mu_oof shape: {mu_oof.shape}  any NaN? {pd.isna(mu_oof).any()}')
print(f'OOF residuals — mean: {residuals_oof.mean():.4f}  std: {residuals_oof.std():.4f}')

In [ ]:
oof_matrix, base_model_names = modl.collect_oof_base_learners(predictor_a)
print(f'oof_matrix shape: {oof_matrix.shape}  (M base learners, N rows)')
print('Base learners:')
for name in base_model_names:
    print(f'  - {name}')

In [ ]:
weights, is_recovery_ok, max_diff = modl.recover_ensemble_weights(
    oof_matrix=oof_matrix,
    mu_oof_ensemble=mu_oof.values,
)

print(f'Recovery max diff: {max_diff:.6e}  ({"OK" if is_recovery_ok else "FALLBACK to uniform"})')
print('\nNon-zero weights:')
for name, w in zip(base_model_names, weights):
    if w > 1e-4:
        print(f'  {name:30s}  {w:.4f}')

# 5. Compute $\hat\mu$ and $\hat\sigma^2_{\text{epist}}$

Weighted statistics over the base learners — coherent with how the WeightedEnsemble combines them. The flag `USE_WEIGHTED_VARIANCE` toggles to uniform statistics as a robustness check.

In [ ]:
mu_recomputed, sigma2_epist_oof = modl.compute_mu_and_epistemic_variance(
    preds_matrix=oof_matrix,
    weights=weights,
    use_weights=USE_WEIGHTED_VARIANCE,
)

print('sigma2_epist_oof:')
print(f'  mean   = {sigma2_epist_oof.mean():.4f}')
print(f'  median = {np.median(sigma2_epist_oof):.4f}')
print(f'  max    = {sigma2_epist_oof.max():.4f}')
print(f'  min    = {sigma2_epist_oof.min():.4f}')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].hist(sigma2_epist_oof, bins=40)
ax[0].set_xlabel('sigma2_epist (OOF)')
ax[0].set_ylabel('count')
ax[0].set_title('Epistemic variance per row')

ax[1].hist(residuals_oof, bins=40)
ax[1].set_xlabel('OOF residual')
ax[1].set_ylabel('count')
ax[1].axvline(0, color='r', linestyle='--')
ax[1].set_title('OOF residuals')
plt.tight_layout()
plt.show()

# 6. Build Aleatoric Targets $\tilde r^2$

Each row gets a noisy point-estimate of the aleatoric variance:

$$\tilde r_i^2 = \max\!\left((y_i - \hat\mu^{\text{OOF}}(\mathbf{x}_i))^2 - \hat\sigma^2_{\text{epist}}(\mathbf{x}_i),\;0\right).$$

Truncation at zero is expected and not an error — see v4 doc §3.7.

In [ ]:
r_tilde_sq, residuals_oof_arr, diag = modl.build_aleatoric_targets(
    y_true=df[TARGET].values,
    mu_oof=mu_oof.values,
    sigma2_epist_oof=sigma2_epist_oof,
)

print('=== Aleatoric target diagnostics ===')
for k, v in diag.items():
    print(f'  {k:25s} = {v}')

# 7. Train Model B — Aleatoric Variance Predictor

Trained on $\log(\tilde r^2 + \epsilon)$ for numerical stability, with the same group-aware splitting as Model A.

In [ ]:
predictor_b = modl.fit_model_b_grouped(
    df=df,
    features=FEATURES,
    r_tilde_sq=r_tilde_sq,
    group_col=GROUP_COL,
    path=os.path.join(PATH_MODELS, PROCESS, 'model_b'),
    presets=PRESETS_B,
    num_bag_folds=NUM_BAG_FOLDS_B,
    num_stack_levels=NUM_STACK_LEVELS_B,
    time_limit=TIME_LIMIT_B,
)

In [ ]:
predictor_b.leaderboard(silent=True)

# 8. Calibration: one-step coverage and scalar recalibration

We test whether the predictive intervals contain the true $y$ on a fraction $\approx \alpha$ of OOF rows. If miscalibrated, we fit a scalar $c$ such that $c \cdot \hat\sigma$ recovers nominal coverage at $\alpha = 0.9$.

In [ ]:
variance_floor = VARIANCE_FLOOR_FRAC * df[TARGET].var()
sigma2_aleat_oof = modl.predict_aleatoric_variance(
    predictor_b, df[FEATURES], variance_floor=variance_floor,
)

sigma2_total_oof = sigma2_epist_oof + sigma2_aleat_oof
sigma_total_oof = np.sqrt(sigma2_total_oof)

y_true = df[TARGET].values
cal_before = modl.calibration_table(
    mu=mu_oof.values, sigma=sigma_total_oof, y_true=y_true,
    alphas=CALIBRATION_ALPHAS,
)
print('=== Calibration BEFORE recalibration ===')
print(cal_before)

In [ ]:
c_opt = modl.fit_recalibration_scalar(
    mu=mu_oof.values,
    sigma=sigma_total_oof,
    y_true=y_true,
    target_alpha=RECALIBRATION_TARGET_ALPHA,
)
print(f'Recalibration scalar c = {c_opt:.4f}')

cal_after = modl.calibration_table(
    mu=mu_oof.values, sigma=c_opt * sigma_total_oof, y_true=y_true,
    alphas=CALIBRATION_ALPHAS,
)
print('\n=== Calibration AFTER recalibration ===')
print(cal_after)

## 8.1 One-step regression metrics

Standard metrics on the OOF predictions, so we can compare directly with the deterministic baseline.

In [ ]:
one_step = evla.one_step_metrics(
    y_true=y_true,
    mu=mu_oof.values,
    sigma=c_opt * sigma_total_oof,
    alphas=CALIBRATION_ALPHAS,
)
print('=== One-step OOF metrics (after recalibration) ===')
print(f'  RMSE = {one_step["rmse"]:.4f}')
print(f'  MAE  = {one_step["mae"]:.4f}')
print(f'  MAPE = {one_step["mape"]:.4f} %')
print(f'  R^2  = {one_step["r2"]:.4f}')
print(f'  Coverage @ alpha=0.9 = {one_step["coverage"][0.9]:.3f}')

# 9. One-Step Inference Example

Apply both models to new rows. This is the **single-pass** mode: every input is assumed to be a real measured row, no recursion. For recursive multi-pass evaluation see the companion notebook.

In [ ]:
preds = modl.predict_with_uncertainty(
    X=df.head(5),
    predictor_a=predictor_a,
    predictor_b=predictor_b,
    weights=weights,
    model_names=base_model_names,
    features=FEATURES,
    variance_floor=variance_floor,
    recalibration_c=c_opt,
    use_weights=USE_WEIGHTED_VARIANCE,
)
preds[['mu', 'sigma_total']].assign(y_true=df[TARGET].head(5).values)

# 10. Persist Artifacts

Save the artifacts needed by the multi-step rollout notebook and by future production inference. The two AutoGluon predictors are saved automatically by AutoGluon to their `path`; only the auxiliary objects need pickling.

In [ ]:
artifacts_path = os.path.join(PATH_MODELS, PROCESS, 'artifacts.pkl')
os.makedirs(os.path.dirname(artifacts_path), exist_ok=True)

artifacts = {
    'features': FEATURES,
    'target': TARGET,
    'group_col': GROUP_COL,
    'base_model_names': base_model_names,
    'weights': weights,
    'variance_floor': variance_floor,
    'recalibration_c': c_opt,
    'use_weighted_variance': USE_WEIGHTED_VARIANCE,
    'calibration_before': cal_before,
    'calibration_after': cal_after,
    'one_step_metrics': one_step,
    'recursive_feature': 'initial_tensile_strength',
    'sort_column': 'pass_number',
}
with open(artifacts_path, 'wb') as f:
    pickle.dump(artifacts, f)

print(f'Saved to: {artifacts_path}')